In [1]:
import scanpy as sc
import pandas as pd
from pathlib import Path

In [3]:
# Paths & basic setup 
adata_path = "/data/projects/oscar/Miranda/results/microglia_subclustering/sub_adata_microglia_leiden6_noOutside_subclustered.h5ad"
adata = sc.read_h5ad(adata_path)

In [6]:
# Name of the clustering column in adata.obs
CLUSTER_COL_ADATA = "leiden_mg_0.4"   # change if your cluster is named differently

# new column name
NEW_COL_NAME = "miranda_submg_cell_type"

# Make sure clusters are strings for consistent mapping
adata.obs[CLUSTER_COL_ADATA] = adata.obs[CLUSTER_COL_ADATA].astype(str).str.strip()

print("Unique clusters in adata:", sorted(adata.obs[CLUSTER_COL_ADATA].unique()))

Unique clusters in adata: ['0', '1', '10', '2', '3', '4', '5', '6', '7', '8', '9']


In [7]:
# Miranda mapping
cluster_to_type = {
    "0":  "Endothelial Cells",
    "1":  "mixture of microglia and neuron",
    "2":  "Lysosomal microglia",
    "3":  "L2/3–L5 IT CTX Glut",
    "4":  "IFN-response microglia (with astrocyte contamination)",
    "5":  "Homeostatic microglia",              
    "6":  "Reactive Astrocyte",
    "7":  "Pvalb GABAergic interneuron",
    "8":  "Phagocytic Microglia/Repair-associated Microglia",
    "9":  "MHC-II+ Activated Microglia",
    "10": "Oligodendrocyte Lineage"
}

# do we have any clusters in adata not covered by Miranda?
adata_clusters = set(adata.obs[CLUSTER_COL_ADATA].unique())
miranda_clusters = set(cluster_to_type.keys())

missing_in_miranda = adata_clusters - miranda_clusters
extra_in_miranda   = miranda_clusters - adata_clusters

print("Clusters in adata not in Miranda mapping:", missing_in_miranda)
print("Clusters in Miranda mapping not in adata:", extra_in_miranda)

Clusters in adata not in Miranda mapping: set()
Clusters in Miranda mapping not in adata: set()


In [8]:
# Attach annotations to adata.obs 
adata.obs[NEW_COL_NAME] = (
    adata.obs[CLUSTER_COL_ADATA]
        .map(cluster_to_type)
        .astype("category")
)

print(adata.obs[[CLUSTER_COL_ADATA, NEW_COL_NAME]].head())
print(adata.obs[NEW_COL_NAME].value_counts().to_frame(name='n_cells'))
print("Cells without a mapped type:",
      adata.obs[NEW_COL_NAME].isna().sum())

                      leiden_mg_0.4      miranda_submg_cell_type
unique_cell_index                                               
new_slide1_aaabbkeo-1             0            Endothelial Cells
new_slide1_aaabegce-1             3          L2/3–L5 IT CTX Glut
new_slide1_aaahghfl-1             5        Homeostatic microglia
new_slide1_aaajccdb-1             7  Pvalb GABAergic interneuron
new_slide1_aaakeflk-1             3          L2/3–L5 IT CTX Glut
                                                    n_cells
miranda_submg_cell_type                                    
Homeostatic microglia                                 15846
Lysosomal microglia                                   15486
mixture of microglia and neuron                       15403
Pvalb GABAergic interneuron                            4625
Phagocytic Microglia/Repair-associated Microglia       2715
L2/3–L5 IT CTX Glut                                    2660
Endothelial Cells                                      2446
Oligo

In [9]:
# by cluster
table = (
adata.obs
.groupby([CLUSTER_COL_ADATA, NEW_COL_NAME])
.size()
.reset_index(name='n_cells')
)
pd.set_option("display.max_rows", None)
display(table)

/tmp/ipykernel_1056411/2349972790.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby([CLUSTER_COL_ADATA, NEW_COL_NAME])


,leiden_mg_0.4,miranda_submg_cell_type,n_cells
0,0,Endothelial Cells,2446
1,0,Homeostatic microglia,0
2,0,IFN-response microglia (with astrocyte contami...,0
3,0,L2/3–L5 IT CTX Glut,0
4,0,Lysosomal microglia,0
5,0,MHC-II+ Activated Microglia,0
6,0,Oligodendrocyte Lineage,0
7,0,Phagocytic Microglia/Repair-associated Microglia,0
8,0,Pvalb GABAergic interneuron,0
9,0,Reactive Astrocyte,0


In [10]:
# Save updated AnnData 
out_h5ad = adata_path.replace(".h5ad", "_with_miranda.h5ad")
adata.write(out_h5ad)
print("Saved updated AnnData with Miranda annotations:", out_h5ad)

Saved updated AnnData with Miranda annotations: /data/projects/oscar/Miranda/results/microglia_subclustering/sub_adata_microglia_leiden6_noOutside_subclustered_with_miranda.h5ad


In [14]:
import os
import pandas as pd
import scanpy as sc

adata_path = "/data/projects/oscar/Miranda/results/microglia_subclustering/sub_adata_microglia_leiden6_noOutside_subclustered_with_miranda.h5ad"
out_dir = "/data/projects/oscar/Miranda/results/microglia_subclustering/results/"
os.makedirs(out_dir, exist_ok=True)

adata = sc.read_h5ad(adata_path)
obs = adata.obs.copy()

# Decide which column identifies slide/run
run_col = None
for c in ["run_key", "batch", "sample_id"]:
    if c in obs.columns:
        run_col = c
        break

if run_col is None:
    # fallback: infer run from obs_names like "new_slide1_12345"
    obs["__run__"] = pd.Index(obs.index).to_series().astype(str).str.split("_", n=1).str[0].values
    run_col = "__run__"

print("Using run column:", run_col)
print("Runs found:", obs[run_col].unique().tolist())

#    Make sure we have a cell_id column for Xenium Explorer
#    Important: Xenium usually wants the *original* cell_id per run,
#    not the namespaced one. We'll strip the "run_" prefix if present.
if "cell_id" in obs.columns:
    cell_id_series = obs["cell_id"].astype(str)
else:
    cell_id_series = pd.Index(obs.index).astype(str)

# If obs_names are like "new_slide1_12345", strip prefix to get "12345"
# (only strip if it looks like it starts with "<run>_")
def strip_prefix(cell_id, run):
    s = str(cell_id)
    pref = f"{run}_"
    return s[len(pref):] if s.startswith(pref) else s

# Export per run
required_col = NEW_COL_NAME ########################################################
if required_col not in obs.columns:
    raise ValueError(f"Missing '{required_col}' in adata.obs. Columns: {obs.columns.tolist()}")

for run in sorted(obs[run_col].astype(str).unique()):
    sub = obs[obs[run_col].astype(str) == run].copy()

    # build per-run cell_id
    sub_cell_id = (sub["cell_id"].astype(str) if "cell_id" in sub.columns else pd.Index(sub.index).astype(str))
    sub_cell_id = [strip_prefix(cid, run) for cid in sub_cell_id]

    export = pd.DataFrame({
        "cell_id": sub_cell_id,
        "group": sub[required_col].astype(str).values
    })

    out_path = os.path.join(out_dir, f"cell_groups_miranda_{NEW_COL_NAME}_{run}.csv")
    export.to_csv(out_path, index=False)
    print("Wrote:", out_path, "| rows:", export.shape[0])

Using run column: run_key
Runs found: ['new_slide1', 'new_slide2']
Wrote: /data/projects/oscar/Miranda/results/microglia_subclustering/results/cell_groups_miranda_miranda_submg_cell_type_new_slide1.csv | rows: 36680
Wrote: /data/projects/oscar/Miranda/results/microglia_subclustering/results/cell_groups_miranda_miranda_submg_cell_type_new_slide2.csv | rows: 30342
